In [ ]:
import json
import re
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import spacy

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

DATA_DIR = Path("../data/processed")
nlp = spacy.load("en_core_web_sm")

## 1. Load extracted events

In [ ]:
def parse_list(value):
    """Lists are stored as JSON text in the CSV (older files used Python-style text)."""
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)) or str(value).strip() == "":
        return []
    try:
        parsed = json.loads(value)
    except (ValueError, TypeError):
        import ast
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [str(value)]
    return parsed if isinstance(parsed, list) else [parsed]


events_df = pd.read_csv(DATA_DIR / "extracted_events_final.csv")
LIST_COLS = ["subject", "object", "agent", "causal_indicators", "quantities",
             "location_entities", "temporal_expressions", "durations"]
for c in LIST_COLS:
    events_df[c] = events_df[c].apply(parse_list)

events_df["event_date"] = pd.to_datetime(events_df["event_date"])
events_df["period_end"] = pd.to_datetime(events_df["period_end"])
events_df["publication_date"] = pd.to_datetime(events_df["publication_date"])
events_df["attributed_to"] = events_df["attributed_to"].fillna("")
events_df["event_time_label"] = events_df["event_time_label"].fillna("")

print("Total events  :", len(events_df))
print("Total articles:", events_df["article_id"].nunique())
events_df[["event_id", "article_id", "event_text", "event_date", "date_confidence"]].head(10)

## 2. Comparable features for every event

* `key_terms` - nouns / proper nouns of the event text (time words removed, a few synonyms unified)
* `numbers`   - the numbers in `quantities` (`118`, `46`, `60` ...)
* `locations` - normalised place names

In [ ]:
CANONICAL = {"rainfall": "rain", "downpour": "rain", "precipitation": "rain",
             "people": "resident", "citizen": "resident", "inhabitant": "resident",
             "millimetre": "mm", "millimeter": "mm", "authority": "official", "officials": "official"}
TIME_TERMS = {"day", "morning", "afternoon", "evening", "night", "noon", "midnight", "overnight",
              "hour", "minute", "week", "month", "year", "a.m.", "p.m.", "am", "pm", "time",
              "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
              "january", "february", "march", "april", "may", "june", "july", "august",
              "september", "october", "november", "december"}

# very small synonym groups for news verbs (extend freely)
VERB_GROUPS = [
    {"open", "set", "establish", "launch", "set up"},
    {"warn", "forecast", "predict", "expect", "caution"},
    {"divert", "redirect", "reroute"},
    {"close", "shut"},
    {"record", "register", "measure", "log", "receive"},
    {"restore", "reconnect", "resume"},
    {"move", "relocate", "evacuate", "shift"},
]


def key_terms(text):
    terms = set()
    for t in nlp(text):
        if t.pos_ in ("NOUN", "PROPN") and not t.is_stop:
            lemma = t.lemma_.lower()
            if lemma in TIME_TERMS:
                continue
            terms.add(CANONICAL.get(lemma, lemma))
    return terms


def numbers_of(quantities):
    nums = set()
    for q in quantities:
        m = re.match(r"\d+(?:\.\d+)?", str(q))
        if m:
            nums.add(float(m.group()))
    return nums


def norm_places(places):
    return {re.sub(r"\s+", " ", p.lower()).strip() for p in places if p}


events_df["key_terms"] = events_df["event_text"].apply(key_terms)
events_df["numbers"] = events_df["quantities"].apply(numbers_of)
events_df["places"] = events_df["location_entities"].apply(norm_places)
events_df["has_clock"] = events_df["event_time_label"].str.contains(r"\d|noon", regex=True)

events_df[["event_id", "event_text", "key_terms", "numbers", "places"]].head(8)

## 3. Embeddings (computed once)
Uses `all-MiniLM-L6-v2` like before. If `sentence-transformers` is not installed the code falls back to TF-IDF
so the notebook still runs - **the printed backend tells you which one was used.**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def embed(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        vecs = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        return vecs, "sentence-transformers (all-MiniLM-L6-v2)"
    except Exception as exc:                                   # offline / not installed
        print("sentence-transformers not available ->", type(exc).__name__)
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.preprocessing import normalize
        mat = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english").fit_transform(texts)
        return normalize(mat).toarray(), "TF-IDF fallback"


embeddings, BACKEND = embed(events_df["event_text"].tolist())
sem_matrix = cosine_similarity(embeddings)
print("Embedding backend:", BACKEND, "| matrix:", sem_matrix.shape)

## 4. Rules that forbid a merge, and the similarity score
Hard rules (never merge): same article, different explicit date, clock times > 2 h apart, opposite polarity
(`No deaths ...`), conflicting numbers (46 vs 60), different action.

In [ ]:
def same_verb_group(a, b):
    if a == b:
        return 1.0
    return 0.8 if any(a in g and b in g for g in VERB_GROUPS) else 0.0


def date_compatible(r1, r2):
    gap = abs((r1["event_date"] - r2["event_date"]).days)
    if r1["date_confidence"] == "explicit" and r2["date_confidence"] == "explicit":
        return gap == 0
    return gap <= 1                       # at least one date was only inferred from the publication date


def time_compatible(r1, r2, max_gap_minutes=120):
    if r1["has_clock"] and r2["has_clock"] and r1["sort_minutes"] == r1["sort_minutes"] and r2["sort_minutes"] == r2["sort_minutes"]:
        return abs(r1["sort_minutes"] - r2["sort_minutes"]) <= max_gap_minutes
    return True


SEM_STRONG = 0.75       # semantic similarity that alone counts as "same action" (MiniLM scale)


def hard_block(r1, r2, i, j):
    """Returns the reason why two events can never be the same event, or '' if they may match."""
    if r1["article_id"] == r2["article_id"]:
        return "same_article"
    if not date_compatible(r1, r2):
        return "different_date"
    if not time_compatible(r1, r2):
        return "different_time"
    if bool(r1["negation"]) != bool(r2["negation"]):
        return "opposite_polarity"
    if r1["numbers"] and r2["numbers"] and not (r1["numbers"] & r2["numbers"]):
        return "conflicting_numbers"
    # same topic is not enough ("shelters were set up" vs "60 residents used the shelters"):
    # the ACTION must agree - same verb / synonym group, the same number, or a very high semantic score
    same_number = bool(r1["numbers"] & r2["numbers"])
    if same_verb_group(r1["action"], r2["action"]) == 0 and not same_number and sem_matrix[i, j] < SEM_STRONG:
        return "different_action"
    return ""


def pair_score(r1, r2, i, j):
    """Weighted mean over the signals that are available for this pair."""
    parts = {}
    parts["semantic"] = (0.40, float(sem_matrix[i, j]))

    a, b = r1["key_terms"], r2["key_terms"]
    if a and b:
        inter = len(a & b)
        parts["terms"] = (0.25, 0.5 * inter / len(a | b) + 0.5 * inter / min(len(a), len(b)))
    else:
        parts["terms"] = (0.25, 0.0)

    parts["verb"] = (0.10, same_verb_group(r1["action"], r2["action"]))

    # positive-only evidence: shared place / shared number (a mismatch is not evidence against)
    p1, p2 = r1["places"], r2["places"]
    if p1 and p2 and (p1 & p2):
        parts["place"] = (0.10, len(p1 & p2) / min(len(p1), len(p2)))
    if r1["numbers"] and r2["numbers"] and (r1["numbers"] & r2["numbers"]):
        parts["number"] = (0.15, 1.0)

    total_w = sum(w for w, _ in parts.values())
    return sum(w * v for w, v in parts.values()) / total_w, {k: round(v, 3) for k, (_, v) in parts.items()}

## 5. Score all cross-article pairs

In [ ]:
rows = []
for i, j in combinations(range(len(events_df)), 2):
    r1, r2 = events_df.iloc[i], events_df.iloc[j]
    reason = hard_block(r1, r2, i, j)
    score, parts = pair_score(r1, r2, i, j)
    rows.append({"i": i, "j": j, "event_id_1": r1["event_id"], "event_id_2": r2["event_id"],
                 "article_1": r1["article_id"], "article_2": r2["article_id"],
                 "text_1": r1["event_text"], "text_2": r2["event_text"],
                 "semantic": round(float(sem_matrix[i, j]), 3), "score": round(score, 3),
                 "blocked_by": reason, "signals": parts})

pairs_df = pd.DataFrame(rows)
cross = pairs_df[pairs_df["article_1"] != pairs_df["article_2"]]
print("Cross-article pairs     :", len(cross))
print("Blocked by a hard rule  :", (cross["blocked_by"] != "").sum())
print(cross[cross["blocked_by"] != ""]["blocked_by"].value_counts().to_string())

## 6. Review the best candidates
`MATCH_THRESHOLD` is the one number to tune. Look at the table below: everything above the line should be a
genuine duplicate; if you see wrong pairs raise it, if obvious duplicates are missing lower it.

In [ ]:
MATCH_THRESHOLD = 0.50

candidates = (pairs_df[(pairs_df["blocked_by"] == "") & (pairs_df["score"] >= MATCH_THRESHOLD - 0.10)]
              .sort_values("score", ascending=False).reset_index(drop=True))
candidates["decision"] = np.where(candidates["score"] >= MATCH_THRESHOLD, "MERGE", "review")

for _, r in candidates.iterrows():
    print(f"{r['decision']:<6} score={r['score']:.3f} sem={r['semantic']:.3f}  #{r['event_id_1']} ({r['article_1']}) <-> #{r['event_id_2']} ({r['article_2']})")
    print(f"        {r['text_1']}")
    print(f"        {r['text_2']}")
    print(f"        signals: {r['signals']}")

## 7. Cluster with average linkage
Pairs are processed from the most to the least similar. Two clusters are joined only when
1. they contain **no two events from the same article**,
2. every cross pair is allowed by the hard rules, and
3. the **average** score between them is >= `MATCH_THRESHOLD`

(this avoids the "A~B, B~C, therefore A=C" chaining of plain union-find used before).

In [ ]:
score_lookup = {(int(r.i), int(r.j)): r.score for r in pairs_df.itertuples()}
block_lookup = {(int(r.i), int(r.j)): r.blocked_by for r in pairs_df.itertuples()}


def pair_key(a, b):
    return (a, b) if a < b else (b, a)


clusters = {i: [i] for i in range(len(events_df))}
label_of = {i: i for i in range(len(events_df))}

for r in candidates[candidates["decision"] == "MERGE"].itertuples():
    ca, cb = label_of[int(r.i)], label_of[int(r.j)]
    if ca == cb:
        continue
    A, B = clusters[ca], clusters[cb]
    articles_a = {events_df.iloc[k]["article_id"] for k in A}
    articles_b = {events_df.iloc[k]["article_id"] for k in B}
    if articles_a & articles_b:
        continue
    cross_pairs = [pair_key(x, y) for x in A for y in B]
    if any(block_lookup[p] for p in cross_pairs):
        continue
    if np.mean([score_lookup[p] for p in cross_pairs]) < MATCH_THRESHOLD:
        continue
    clusters[ca] = A + B
    for k in B:
        label_of[k] = ca
    del clusters[cb]

multi = {c: m for c, m in clusters.items() if len(m) > 1}
print("Events before merging:", len(events_df))
print("Timeline entries after:", len(clusters))
print("Merged groups:", len(multi))
for c, members in multi.items():
    print("-" * 100)
    for k in members:
        r = events_df.iloc[k]
        print(f"  #{r['event_id']:<3} {r['article_id']}  {r['event_text']}")

## 8. Build one record per cluster
* representative text = the most informative member (subject, object, place, explicit date/time)
* date  = explicit date of the members if one exists, otherwise the earliest inferred date
* time  = earliest clock time of members on that date

In [ ]:
def unique(seq):
    out, seen = [], set()
    for x in seq:
        key = str(x).strip().lower()
        if key and key not in seen:
            seen.add(key)
            out.append(str(x).strip())
    return out


def info_score(r):
    return (bool(r["subject"]) + bool(r["object"]) + bool(r["location_entities"])
            + (r["date_confidence"] == "explicit") + bool(r["event_time_label"]) + len(r["event_text"].split()) / 50)


merged_rows = []
for cid, members in sorted(clusters.items(), key=lambda kv: kv[0]):
    grp = events_df.iloc[members].copy()
    grp["_info"] = grp.apply(info_score, axis=1)
    rep = grp.sort_values(["_info", "article_id"], ascending=[False, True]).iloc[0]

    explicit = grp[grp["date_confidence"] == "explicit"]
    chosen_date = (explicit["event_date"].min() if len(explicit) else grp["event_date"].min())
    on_day = grp[grp["event_date"] == chosen_date]
    timed = on_day[on_day["sort_minutes"].notna()]
    if len(timed):
        t_row = timed.sort_values(["has_clock", "sort_minutes"], ascending=[False, True]).iloc[0]
        minutes, label = float(t_row["sort_minutes"]), t_row["event_time_label"]
    else:
        minutes, label = np.nan, ""

    first = grp.sort_values(["publication_date", "article_id", "sentence_id"]).iloc[0]
    merged_rows.append({
        "event_id": len(merged_rows) + 1,
        "event_text": rep["event_text"],
        "event_date": chosen_date,
        "event_minutes": minutes,
        "event_time_label": label,
        "date_confidence": "explicit" if len(explicit) else grp["date_confidence"].iloc[0],
        "date_source": rep["date_source"] if rep["event_date"] == chosen_date else grp.iloc[0]["date_source"],
        "period_end": grp["period_end"].max(),
        "article_ids": unique(grp["article_id"]),
        "source_event_ids": [int(x) for x in grp["event_id"]],
        "source_texts": [f"{a}: {t}" for a, t in zip(grp["article_id"], grp["event_text"])],
        "actions": unique(grp["action"]),
        "subjects": unique(s for lst in grp["subject"] for s in lst),
        "objects": unique(o for lst in grp["object"] for o in lst),
        "locations": unique(l for lst in grp["location_entities"] for l in lst),
        "attributed_to": unique(grp["attributed_to"]),
        "quantities": unique(q for lst in grp["quantities"] for q in lst),
        "negation": bool(grp["negation"].any()),
        "event_count": len(grp),
        "article_count": grp["article_id"].nunique(),
        "first_reported": first["publication_date"],
        "rep_article": rep["article_id"],
        "rep_sentence": int(rep["sentence_id"]),
        "rep_trigger": int(rep["trigger_index"]),
    })

merged_df = pd.DataFrame(merged_rows)
print("Merged event records:", len(merged_df))
merged_df[["event_id", "event_date", "event_time_label", "event_text", "article_ids", "date_confidence"]]

## 9. Checks

In [ ]:
assert merged_df["article_ids"].apply(len).eq(merged_df["event_count"]).all(), \
    "a merged event contains two events of the same article"
print("OK - no merged event contains two events from the same article")
print("Events reported by more than one article:", (merged_df["article_count"] > 1).sum())

## 10. Save

In [ ]:
out = merged_df.copy()
for c in ["article_ids", "source_event_ids", "source_texts", "actions", "subjects", "objects",
          "locations", "attributed_to", "quantities"]:
    out[c] = out[c].apply(json.dumps)
for c in ["event_date", "period_end", "first_reported"]:
    out[c] = out[c].dt.strftime("%Y-%m-%d")
out.to_csv(DATA_DIR / "merged_events_final.csv", index=False)

pairs_out = pairs_df[pairs_df["article_1"] != pairs_df["article_2"]].drop(columns=["i", "j"]).copy()
pairs_out["signals"] = pairs_out["signals"].apply(json.dumps)
pairs_out.sort_values("score", ascending=False).to_csv(DATA_DIR / "event_matching_pairs_final.csv", index=False)

print("Saved", DATA_DIR / "merged_events_final.csv")
print("Saved", DATA_DIR / "event_matching_pairs_final.csv")